In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from model import MazeTransformer
import torch
import yaml

# STATE_DICT_PATH = './models/big_rotard_tuning_more_data'
# with open("./configs/config.yaml", "r") as f:
#     config = yaml.safe_load(f)

# model = MazeTransformer(config["model"])
# model.load_state_dict(torch.load(STATE_DICT_PATH, weights_only=True))
# model.eval()

In [3]:
# from train import MazeDataset, create_causal_mask
# import os

# train_dataset = MazeDataset(os.path.join("./data", "train"))
# sequences, targets, sizes = train_dataset[100]

# sequences = sequences.unsqueeze(0)
# targets = targets.unsqueeze(0)
# sizes = sizes.unsqueeze(0)

# seq_len = sequences.shape[1]
# masks = create_causal_mask(sizes, seq_len)

# sequences = sequences.to(config["model"]["device"])
# masks = masks.to(config["model"]["device"])

# model_out = model.forward(sequences, masks).squeeze()

# sz = sizes.item()
# predictions = model_out.squeeze()[sz - 1: sz - 1 + targets[0].shape[0]]

In [4]:
# print("Predicted:", torch.argmax(predictions, dim=-1))
# print("Target:   ", targets.squeeze())

In [27]:
# model.forward(sequences[:,:sz].to('mps'), torch.ones((sz, sz)).to('mps')).squeeze()[-1]

In [34]:
ex_input = torch.IntTensor([1, 2, 3, 4, 5])
mask = torch.ones((5, 5)).tril()
mask[:4, :4] = True
mask = mask.bool()

model = MazeTransformer({
    "vocab_size" : 10,
    "output_vocab_size" : 5,
    "device" : "cpu",
    "d_model" : 100,
    "max_seq_len" : 50,
    "n_heads" : 1,
    "n_layers" : 1,
    "dropout" : 0
})
model.eval()

with torch.no_grad():
    print(model.forward(ex_input.unsqueeze(0), mask.unsqueeze(0)))

Attention scores:
tensor([[[ 0.5518,  0.7837,  0.3362, -0.3567, -0.9434],
         [-0.3389,  0.0095, -0.7331,  0.4573,  0.4565],
         [ 0.5231, -0.6798, -1.5724, -0.2711, -0.0801],
         [ 0.3224, -0.1970, -1.3397,  0.4689, -0.5665],
         [ 0.1674, -0.0022, -0.8016,  0.2413,  1.3633]]])


In [35]:
truncated_ex_input = ex_input[:-1]
truncated_mask = mask[:-1, :-1]

with torch.no_grad():
    print(model.forward(truncated_ex_input.unsqueeze(0), truncated_mask.unsqueeze(0)))

Attention scores:
tensor([[[ 0.5518,  0.7837,  0.3362, -0.3567, -0.9434],
         [-0.3389,  0.0095, -0.7331,  0.4573,  0.4565],
         [ 0.5231, -0.6798, -1.5724, -0.2711, -0.0801],
         [ 0.3224, -0.1970, -1.3397,  0.4689, -0.5665]]])


In [33]:
mask.bool().dtype

torch.bool

In [129]:
sz

73

In [98]:
predictions

tensor([[-5.7898, -3.7477, -1.7164,  8.7442, -0.8414],
        [ 0.6164, -6.0528, -3.3924,  5.2317, -2.3537],
        [ 8.6922, -3.6795, -4.8091, -1.4878, -2.8170],
        [-5.1412, -0.8244, -3.2744,  8.6522, -1.9512],
        [ 0.6388, -4.8175, -4.5454,  6.5526, -3.5797],
        [ 9.4590, -3.1947, -4.8296, -2.0000, -2.7347],
        [ 9.5940, -2.3424, -5.0030, -1.9510, -3.0558],
        [-2.6889, -4.5642, -2.5633,  8.1937, -3.0290],
        [-2.1688, -5.4613, -3.4617,  9.0006, -3.2404],
        [-1.9887, -3.7596, -4.3496,  8.5792, -3.1745],
        [ 9.1022, -3.0898, -5.1672, -1.1496, -2.8942],
        [ 9.3340, -1.3293, -1.7275, -5.0421, -3.2828],
        [ 8.7263, -1.2884, -0.8830, -5.7929, -3.0174],
        [ 8.4074, -1.6718, -2.2796, -5.5059, -1.2769],
        [-2.3926, -3.1503, -3.2922,  1.4071,  7.1604]], device='mps:0',
       grad_fn=<SliceBackward0>)

In [114]:
model_out[72]

tensor([-5.7898, -3.7477, -1.7164,  8.7442, -0.8414], device='mps:0',
       grad_fn=<SelectBackward0>)

In [123]:
ex_in = sequences[0][:sizes[0]].to('mps')
generated_path = model.generate(ex_in, max_tokens=100)
generated_path

tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [76]:
# THERE'S SOMETHING WEIRD HERE -> is the model not being trained on the right mask?
# Try filling mask with 1s and then do a model.forward on just the prefix

tensor([3, 3, 0, 3, 0, 3, 0, 3, 0, 3, 0, 2, 0, 3, 1, 2, 0, 3, 4],
       device='mps:0')

In [77]:
targets[0]

tensor([3, 3, 3, 0, 3, 0, 0, 0, 2, 0, 2, 2, 0, 0, 3, 3, 3, 1, 3, 1, 3, 3, 0, 0,
        4])